In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings

warnings.filterwarnings("ignore")

In [ ]:
glass = pd.read_csv("Glass.csv")
X = glass.drop('Type', axis=1)
le = LabelEncoder()
y = le.fit_transform( glass['Type'] )
y

In [ ]:
le.classes_

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25,
                                                    stratify=y)

## Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

# Compare penalties using SAGA solver

In [ ]:
solver = 'saga'

penalties = [
    'l2',
    'l1',
    'elasticnet',
    None
]

scores = []


for p in penalties:

    if p == 'elasticnet':

        lr = LogisticRegression(
            solver=solver,
            penalty=p,
            l1_ratio=0.5,
            max_iter=5000,
            random_state=25
        )

    else:

        lr = LogisticRegression(
            solver=solver,
            penalty=p,
            max_iter=5000,
            random_state=25
        )


    lr.fit(X_train_scaled,y_train)
    y_pred = lr.predict(X_test_scaled)
    scores.append([p,accuracy_score(y_test, y_pred)])

df_scores = pd.DataFrame(scores,columns=['Penalty','Accuracy'])
print(df_scores.sort_values('Accuracy',ascending=False))


In [ ]:
solvers = [
    'sag',
    'newton-cg',
    'newton-cholesky',
    'lbfgs'
]


penalties = [
    'l2',
    None
]
scores = []
for p in penalties:

    for s in solvers:


        lr = LogisticRegression(
            solver=s,
            penalty=p,
            max_iter=5000,
            random_state=25
        )


        lr.fit(
            X_train_scaled,
            y_train
        )


        y_pred = lr.predict(
            X_test_scaled
        )


        scores.append(
            [
                p,
                s,
                accuracy_score(
                    y_test,
                    y_pred
                )
            ]
        )

df_scores = pd.DataFrame(
    scores,
    columns=[
        'Penalty',
        'Solver',
        'Accuracy'
    ]
)

print(
    df_scores.sort_values(
        'Accuracy',
        ascending=False
    )
)


In [ ]:
lr = LogisticRegression(solver='newton-cg',penalty='l2',max_iter=5000,random_state=25)
lr.fit(X_train_scaled,y_train)
# Prediction probability
y_pred_prob = lr.predict_proba(X_test_scaled)
print(y_pred_prob)

In [ ]:
# Prediction
y_pred = lr.predict(X_test_scaled)
print(y_pred)
# Accuracy
print(accuracy_score(y_test,y_pred))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test,y_pred)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues', xticklabels=le.classes_,yticklabels=le.classes_)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Logistic Regression")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.show()

In [ ]:
# Classification Report with class names

print(classification_report(y_test,y_pred,target_names=le.classes_))